# Stage 2: Classical Multilayer Perceptron (MLP) Baseline (Model 1)

**Project Title:** Adaptive Hybrid Quantum-Classical Neural Network for Medical Disease Prediction Using Dynamic Quantum Layer Selection and Explainable AI  
**Focus:** Establishing a Fair, Rigorous, Leakage-Safe Classical Deep Learning Benchmark Across All Three Clinical Tasks

---

## 1. Stage Overview & Purpose
In this notebook, we implement and evaluate **Model 1: Classical Multilayer Perceptron (MLP)**.
The classical MLP serves as the primary classical deep learning reference baseline against which the Fixed HQNN (Model 2) and Adaptive HQNN (Model 3) will be evaluated.

### Architectural Principles & Experimental Fairness
* **Controlled Complexity**: The MLP uses a compact 2-hidden-layer feedforward topology with Layer Normalization, Dropout ($p=0.2$), and ReLU activations to prevent overfitting on small clinical sample sizes ($N=303$ to $768$).
* **Class-Weighted Loss**: Loss functions employ dynamic positive-class weighting ($	ext{pos\_weight}$) computed exclusively on each training fold to handle medical class imbalances.
* **Leakage-Safe 5-Fold Stratified Cross-Validation**: We strictly utilize the split partitions established in Stage 1 (`data/splits/`), guaranteeing zero leakage during preprocessing and evaluation.
* **Comprehensive Clinical Metrics**: We record Accuracy, Precision, Recall (Sensitivity), Specificity, F1-score, ROC-AUC, and Confusion Matrices across all CV folds.


## 2. Environment Setup & Dependency Installation
Run the cell below to load PyTorch, scikit-learn, and visualization packages.


In [1]:
# Install dependencies if running in Google Colab
# !pip install -q torch scikit-learn pandas numpy matplotlib seaborn

import os
import sys
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Global visualization settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Deterministic Seed Configuration
RANDOM_STATE = 42

def set_deterministic_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_deterministic_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Execution device: {device}")
print(f"Deterministic seed locked to: {RANDOM_STATE}")


PyTorch version: 2.6.0+cpu
Execution device: cpu
Deterministic seed locked to: 42


## 3. Project Directory Verification & Path Management


In [2]:
# Determine workspace root
if os.path.exists("notebooks") or os.path.exists("data"):
    PROJECT_DIR = os.path.abspath(".")
elif os.path.exists("../data") or os.path.basename(os.getcwd()) == 'notebooks':
    PROJECT_DIR = os.path.abspath("..")
elif os.path.exists(r"C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn"):
    PROJECT_DIR = r"C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn"
else:
    PROJECT_DIR = os.path.abspath(".")

DATA_RAW_DIR = os.path.join(PROJECT_DIR, 'data', 'raw')
DATA_SPLITS_DIR = os.path.join(PROJECT_DIR, 'data', 'splits')
RESULTS_METRICS_DIR = os.path.join(PROJECT_DIR, 'results', 'metrics')
RESULTS_MODELS_DIR = os.path.join(PROJECT_DIR, 'results', 'models', 'classical_mlp')
RESULTS_FIGURES_DIR = os.path.join(PROJECT_DIR, 'results', 'figures')
RESULTS_TABLES_DIR = os.path.join(PROJECT_DIR, 'results', 'tables')

for d in [RESULTS_METRICS_DIR, RESULTS_MODELS_DIR, RESULTS_FIGURES_DIR, RESULTS_TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directories verified at:", PROJECT_DIR)


Project directories verified at: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn


## 4. Dataset Loader & Leakage-Safe Preprocessing Pipeline
We reload the raw datasets and strictly apply the partition indices generated in Stage 1.


In [3]:
def load_dataset_data(dataset_key):
    """Load raw dataset and split configuration for a given dataset key."""
    split_path = os.path.join(DATA_SPLITS_DIR, dataset_key, "splits_config.json")
    with open(split_path, "r") as f:
        split_info = json.load(f)
        
    if dataset_key == "breast_cancer":
        raw_file = os.path.join(DATA_RAW_DIR, "breast_cancer", "wdbc.data")
        feature_names = [
            'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean',
            'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
            'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
            'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se',
            'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
            'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst'
        ]
        raw_cols = ['id', 'diagnosis'] + feature_names
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols)
        df = df_raw.drop(columns=['id']).copy()
        df['target'] = df['diagnosis'].map({'B': 0, 'M': 1}).astype(int)
        df = df.drop(columns=['diagnosis'])
        
    elif dataset_key == "heart":
        raw_file = os.path.join(DATA_RAW_DIR, "heart", "processed.cleveland.data")
        feature_names = [
            'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
            'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
        ]
        raw_cols = feature_names + ['num']
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols, na_values='?')
        df = df_raw.copy()
        df['target'] = (df['num'] > 0).astype(int)
        df = df.drop(columns=['num'])
        
    elif dataset_key == "diabetes":
        raw_file = os.path.join(DATA_RAW_DIR, "diabetes", "pima-indians-diabetes.csv")
        feature_names = [
            'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
        ]
        raw_cols = feature_names + ['Outcome']
        df_raw = pd.read_csv(raw_file, header=None, names=raw_cols)
        df = df_raw.copy()
        df['target'] = df['Outcome'].astype(int)
        df = df.drop(columns=['Outcome'])
        for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
            df[col] = df[col].replace(0, np.nan)
            
    return df, feature_names, split_info

class FoldPreprocessor:
    """Leakage-safe fold transformer fitting strictly on training feature matrix."""
    def __init__(self):
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()
        
    def fit(self, X_train):
        self.imputer.fit(X_train)
        X_imp = self.imputer.transform(X_train)
        self.scaler.fit(X_imp)
        return self
        
    def transform(self, X):
        X_imp = self.imputer.transform(X)
        return self.scaler.transform(X_imp)
        
    def fit_transform(self, X_train):
        return self.fit(X_train).transform(X_train)

print("Data loader and FoldPreprocessor ready.")


Data loader and FoldPreprocessor ready.


## 5. Classical Multilayer Perceptron (MLP) Architecture
We specify a 2-hidden-layer feedforward network with:
* Linear projection: $D 	o H_1$ ($32$ units)
* Layer Normalization
* ReLU activation
* Dropout ($p = 0.20$)
* Linear projection: $H_1 	o H_2$ ($16$ units)
* ReLU activation
* Output projection: $H_2 	o 1$ (Raw binary logits)

$$\hat{y}_{logit} = W_3 \cdot 	ext{ReLU}\left(W_2 \cdot 	ext{Dropout}\left(	ext{ReLU}\left(	ext{LayerNorm}(W_1 x + b_1)ight)ight) + b_2ight) + b_3$$
$$P(y=1|x) = \sigma(\hat{y}_{logit}) = rac{1}{1 + e^{-\hat{y}_{logit}}}$$


In [4]:
class ClassicalMLP(nn.Module):
    """
    Classical Multi-Layer Perceptron (MLP) Baseline for Binary Medical Prediction.
    Architecture: Input(D) -> Dense(32) -> LayerNorm -> ReLU -> Dropout(0.2) -> Dense(16) -> ReLU -> Dense(1)
    """
    def __init__(self, input_dim, hidden_dim=32, dropout_rate=0.2):
        super(ClassicalMLP, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.relu2 = nn.ReLU()
        
        self.out = nn.Linear(hidden_dim // 2, 1)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu2(x)
        logits = self.out(x)
        return logits.squeeze(-1)

# Inspect model parameter counts across all three datasets
print("Classical MLP Model Architectures & Parameter Counts:")
for ds, feat_count in [('WDBC Breast Cancer', 30), ('UCI Heart Disease', 13), ('Pima Diabetes', 8)]:
    m = ClassicalMLP(input_dim=feat_count)
    total_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"- {ds:25s} (D={feat_count:2d}): Total Trainable Parameters = {total_params:,}")


Classical MLP Model Architectures & Parameter Counts:
- WDBC Breast Cancer        (D=30): Total Trainable Parameters = 1,601
- UCI Heart Disease         (D=13): Total Trainable Parameters = 1,057
- Pima Diabetes             (D= 8): Total Trainable Parameters = 897


## 6. Training Engine & Medical Metric Evaluation

### Medical Metrics Evaluated:
1. **Accuracy**: Overall fraction of correct predictions: $rac{TP + TN}{TP + TN + FP + FN}$
2. **Precision**: Positive predictive value: $rac{TP}{TP + FP}$
3. **Recall (Sensitivity)**: True positive rate (vital in oncology/cardiology): $rac{TP}{TP + FN}$
4. **Specificity**: True negative rate (avoiding unnecessary invasive interventions): $rac{TN}{TN + FP}$
5. **F1-Score**: Harmonic mean of precision and sensitivity: $rac{2 \cdot 	ext{Precision} \cdot 	ext{Recall}}{	ext{Precision} + 	ext{Recall}}$
6. **ROC-AUC**: Area under the Receiver Operating Characteristic curve.

### Loss Function & Imbalance Handling:
$$\mathcal{L}_{BCE} = -rac{1}{N} \sum_{i=1}^N \left[ w_{pos} \cdot y_i \log \sigma(z_i) + (1 - y_i) \log (1 - \sigma(z_i)) ight], \quad w_{pos} = rac{N_{neg}}{N_{pos}}$$


In [5]:
def compute_medical_metrics(y_true, y_probs, threshold=0.5):
    """Compute full suite of medical classification metrics."""
    y_true = np.array(y_true)
    y_probs = np.array(y_probs)
    y_pred = (y_probs >= threshold).astype(int)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Specificity = TN / (TN + FP)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    try:
        auc = roc_auc_score(y_true, y_probs)
    except Exception:
        auc = 0.5
        
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "specificity": spec,
        "f1": f1,
        "roc_auc": auc,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }

def train_mlp_fold(model, train_loader, val_loader, pos_weight, lr=0.003, weight_decay=1e-4, max_epochs=120, patience=18):
    """Train PyTorch MLP on a single fold with early stopping and class-weighted loss."""
    model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=6)
    
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(max_epochs):
        model.train()
        train_loss_acc = 0.0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss_acc += loss.item() * len(batch_y)
            
        train_loss = train_loss_acc / len(train_loader.dataset)
        
        # Validation evaluation
        model.eval()
        val_loss_acc = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                val_loss_acc += loss.item() * len(batch_y)
                
        val_loss = val_loss_acc / len(val_loader.dataset)
        scheduler.step(val_loss)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
                
    if best_state is not None:
        model.load_state_dict(best_state)
        
    return model, history

print("Training engine functions successfully defined.")


Training engine functions successfully defined.


## 7. 5-Fold Stratified Cross-Validation Execution
We now run the full cross-validation benchmark for **Model 1: Classical MLP** across WDBC Breast Cancer, UCI Cleveland Heart Disease, and Pima Indians Diabetes.


In [6]:
cv_results_all = []
oof_predictions_dict = {}
test_predictions_dict = {}

BATCH_SIZE = 32
LR = 0.003
EPOCHS = 150
PATIENCE = 20

for ds_key in ['breast_cancer', 'heart', 'diabetes']:
    set_deterministic_seed(RANDOM_STATE)
    df, feature_names, split_info = load_dataset_data(ds_key)
    ds_name = split_info["display_name"]
    print(f"\n{'='*65}")
    print(f"Running 5-Fold Cross-Validation for: {ds_name}")
    print(f"{'='*65}")
    
    dev_idx = np.array(split_info["dev_indices"])
    test_idx = np.array(split_info["test_indices"])
    
    X_dev = df.iloc[dev_idx][feature_names].values
    y_dev = df.iloc[dev_idx]['target'].values
    X_test = df.iloc[test_idx][feature_names].values
    y_test = df.iloc[test_idx]['target'].values
    
    oof_probs = np.zeros(len(dev_idx))
    test_fold_probs = np.zeros((5, len(test_idx)))
    
    fold_metrics = []
    
    for fold_info in split_info["cv_folds"]:
        fold_num = fold_info["fold"]
        train_sub_idx = np.array(fold_info["train_dev_idx"])
        val_sub_idx = np.array(fold_info["val_dev_idx"])
        
        # 1. Leakage-safe fold transformation
        preprocessor = FoldPreprocessor()
        X_train_scaled = preprocessor.fit_transform(X_dev[train_sub_idx])
        X_val_scaled = preprocessor.transform(X_dev[val_sub_idx])
        X_test_scaled = preprocessor.transform(X_test)
        
        y_train = y_dev[train_sub_idx]
        y_val = y_dev[val_sub_idx]
        
        # Compute training fold class weight
        n_pos = np.sum(y_train == 1)
        n_neg = np.sum(y_train == 0)
        pos_weight = float(n_neg / n_pos) if n_pos > 0 else 1.0
        
        # PyTorch DataLoaders
        train_ds = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
        val_ds = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
        
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
        
        # Initialize and train MLP
        set_deterministic_seed(RANDOM_STATE + fold_num)
        model = ClassicalMLP(input_dim=len(feature_names), hidden_dim=32, dropout_rate=0.2)
        
        t0 = time.time()
        model, history = train_mlp_fold(
            model, train_loader, val_loader, pos_weight=pos_weight,
            lr=LR, weight_decay=1e-4, max_epochs=EPOCHS, patience=PATIENCE
        )
        elapsed = time.time() - t0
        
        # Predict on validation fold
        model.eval()
        with torch.no_grad():
            val_t = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
            val_logits = model(val_t)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
            
            test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
            test_logits = model(test_t)
            t_probs = torch.sigmoid(test_logits).cpu().numpy()
            
        oof_probs[val_sub_idx] = val_probs
        test_fold_probs[fold_num - 1] = t_probs
        
        # Compute fold metrics
        metrics = compute_medical_metrics(y_val, val_probs)
        metrics["dataset"] = ds_key
        metrics["dataset_name"] = ds_name
        metrics["model"] = "Classical MLP (M1)"
        metrics["fold"] = fold_num
        metrics["runtime_sec"] = elapsed
        
        fold_metrics.append(metrics)
        cv_results_all.append(metrics)
        
        # Save model checkpoint
        ckpt_path = os.path.join(RESULTS_MODELS_DIR, f"mlp_{ds_key}_fold_{fold_num}.pt")
        torch.save(model.state_dict(), ckpt_path)
        
        print(f"Fold {fold_num}/5: Acc={metrics['accuracy']*100:.2f}%, Rec={metrics['recall']*100:.2f}%, Spec={metrics['specificity']*100:.2f}%, F1={metrics['f1']*100:.2f}%, AUC={metrics['roc_auc']:.4f} ({elapsed:.2f}s)")
        
    # Aggregate CV metrics
    df_ds_cv = pd.DataFrame(fold_metrics)
    print(f"--> {ds_name} 5-Fold Mean: "
          f"Acc = {df_ds_cv['accuracy'].mean()*100:.2f} ± {df_ds_cv['accuracy'].std()*100:.2f}%, "
          f"Rec = {df_ds_cv['recall'].mean()*100:.2f} ± {df_ds_cv['recall'].std()*100:.2f}%, "
          f"Spec = {df_ds_cv['specificity'].mean()*100:.2f} ± {df_ds_cv['specificity'].std()*100:.2f}%, "
          f"F1 = {df_ds_cv['f1'].mean()*100:.2f} ± {df_ds_cv['f1'].std()*100:.2f}%, "
          f"AUC = {df_ds_cv['roc_auc'].mean():.4f} ± {df_ds_cv['roc_auc'].std():.4f}")
          
    # Ensemble Test Evaluation
    ensemble_test_probs = np.mean(test_fold_probs, axis=0)
    test_metrics = compute_medical_metrics(y_test, ensemble_test_probs)
    test_metrics["dataset"] = ds_key
    test_metrics["dataset_name"] = ds_name
    test_metrics["model"] = "Classical MLP (M1) - Test Ensemble"
    test_metrics["fold"] = "TEST_ENSEMBLE"
    cv_results_all.append(test_metrics)
    
    print(f"--> Holdout Test (N={len(y_test)}): Acc={test_metrics['accuracy']*100:.2f}%, Rec={test_metrics['recall']*100:.2f}%, Spec={test_metrics['specificity']*100:.2f}%, F1={test_metrics['f1']*100:.2f}%, AUC={test_metrics['roc_auc']:.4f}")
    
    oof_predictions_dict[ds_key] = {"y_true": y_dev.tolist(), "oof_probs": oof_probs.tolist()}
    test_predictions_dict[ds_key] = {"y_true": y_test.tolist(), "test_probs": ensemble_test_probs.tolist()}


Running 5-Fold Cross-Validation for: Wisconsin Diagnostic Breast Cancer (WDBC)
Fold 1/5: Acc=96.70%, Rec=94.12%, Spec=98.25%, F1=95.52%, AUC=0.9974 (3.77s)
Fold 2/5: Acc=97.80%, Rec=100.00%, Spec=96.49%, F1=97.14%, AUC=0.9990 (0.76s)
Fold 3/5: Acc=98.90%, Rec=100.00%, Spec=98.25%, F1=98.55%, AUC=0.9995 (0.80s)
Fold 4/5: Acc=97.80%, Rec=97.06%, Spec=98.25%, F1=97.06%, AUC=0.9794 (0.72s)
Fold 5/5: Acc=97.80%, Rec=97.06%, Spec=98.25%, F1=97.06%, AUC=0.9985 (0.99s)
--> Wisconsin Diagnostic Breast Cancer (WDBC) 5-Fold Mean: Acc = 97.80 ± 0.78%, Rec = 97.65 ± 2.46%, Spec = 97.89 ± 0.78%, F1 = 97.07 ± 1.07%, AUC = 0.9947 ± 0.0086
--> Holdout Test (N=114): Acc=99.12%, Rec=97.62%, Spec=100.00%, F1=98.80%, AUC=0.9907
Running 5-Fold Cross-Validation for: UCI Heart Disease (Cleveland)
Fold 1/5: Acc=85.71%, Rec=86.96%, Spec=84.62%, F1=85.11%, AUC=0.9264 (0.56s)
Fold 2/5: Acc=85.71%, Rec=77.27%, Spec=92.59%, F1=82.93%, AUC=0.8805 (0.60s)
Fold 3/5: Acc=81.25%, Rec=77.27%, Spec=84.62%, F1=79.07%, AUC=

## 8. Export Benchmark Metrics & Generate Publication Summary


In [7]:
df_cv_all = pd.DataFrame(cv_results_all)

# Save full metrics log to results/metrics/
metrics_csv_path = os.path.join(RESULTS_METRICS_DIR, "m1_classical_mlp_metrics.csv")
df_cv_all.to_csv(metrics_csv_path, index=False)
print(f"[OK] Full M1 Classical MLP metrics saved to: {metrics_csv_path}")

# Build formatted Table 2 slice for Model 1
summary_rows = []
for ds_key in ['breast_cancer', 'heart', 'diabetes']:
    df_sub = df_cv_all[(df_cv_all['dataset'] == ds_key) & (df_cv_all['fold'] != 'TEST_ENSEMBLE')]
    df_test = df_cv_all[(df_cv_all['dataset'] == ds_key) & (df_cv_all['fold'] == 'TEST_ENSEMBLE')].iloc[0]
    
    ds_name = df_sub.iloc[0]['dataset_name']
    
    summary_rows.append({
        "Dataset": ds_name,
        "Model": "Classical MLP (M1)",
        "CV Accuracy (%)": f"{df_sub['accuracy'].mean()*100:.2f} ± {df_sub['accuracy'].std()*100:.2f}",
        "CV Precision (%)": f"{df_sub['precision'].mean()*100:.2f} ± {df_sub['precision'].std()*100:.2f}",
        "CV Recall (%)": f"{df_sub['recall'].mean()*100:.2f} ± {df_sub['recall'].std()*100:.2f}",
        "CV Specificity (%)": f"{df_sub['specificity'].mean()*100:.2f} ± {df_sub['specificity'].std()*100:.2f}",
        "CV F1 (%)": f"{df_sub['f1'].mean()*100:.2f} ± {df_sub['f1'].std()*100:.2f}",
        "CV ROC-AUC": f"{df_sub['roc_auc'].mean():.4f} ± {df_sub['roc_auc'].std():.4f}",
        "Holdout Test AUC": f"{df_test['roc_auc']:.4f}",
        "Holdout Test F1 (%)": f"{df_test['f1']*100:.2f}"
    })

df_mlp_summary = pd.DataFrame(summary_rows)
table_mlp_path = os.path.join(RESULTS_TABLES_DIR, "table_m1_classical_mlp_summary.csv")
df_mlp_summary.to_csv(table_mlp_path, index=False)

display(df_mlp_summary)


[OK] Full M1 Classical MLP metrics saved to: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn\results\metrics\m1_classical_mlp_metrics.csv
                                     Dataset  ... Holdout Test F1 (%)
0  Wisconsin Diagnostic Breast Cancer (WDBC)  ...               98.80
1              UCI Heart Disease (Cleveland)  ...               86.67
2             Pima Indians Diabetes Database  ...               67.72
[3 rows x 10 columns]


## 9. Publication Visualizations: ROC Curves & Confusion Matrices


In [8]:
# Figure: ROC Curves for 5-Fold OOF and Holdout Test across all 3 datasets
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dataset_colors = {'breast_cancer': '#1b9e77', 'heart': '#d95f02', 'diabetes': '#7570b3'}

for idx, ds_key in enumerate(['breast_cancer', 'heart', 'diabetes']):
    oof_data = oof_predictions_dict[ds_key]
    test_data = test_predictions_dict[ds_key]
    
    y_dev = oof_data["y_true"]
    oof_p = oof_data["oof_probs"]
    y_test = test_data["y_true"]
    test_p = test_data["test_probs"]
    
    # OOF ROC
    fpr_oof, tpr_oof, _ = roc_curve(y_dev, oof_p)
    auc_oof = roc_auc_score(y_dev, oof_p)
    
    # Test ROC
    fpr_test, tpr_test, _ = roc_curve(y_test, test_p)
    auc_test = roc_auc_score(y_test, test_p)
    
    ax = axes[idx]
    ax.plot(fpr_oof, tpr_oof, label=f'5-Fold OOF (AUC = {auc_oof:.3f})', color=dataset_colors[ds_key], lw=2.2)
    ax.plot(fpr_test, tpr_test, label=f'Holdout Test (AUC = {auc_test:.3f})', color='black', lw=1.8, linestyle='--')
    ax.plot([0, 1], [0, 1], color='gray', linestyle=':', lw=1.2)
    
    ds_name = DATASET_CONFIG[ds_key]["display_name"] if 'DATASET_CONFIG' in globals() else ds_key.replace('_', ' ').title()
    ax.set_title(f"{ds_name}\n(Classical MLP Baseline)", fontweight='bold')
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.legend(loc='lower right', frameon=True)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])

plt.suptitle('Figure 2.1: ROC Curves for Classical MLP Baseline (M1)', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
fig_roc_path = os.path.join(RESULTS_FIGURES_DIR, "fig_stage2_mlp_roc_curves.png")
plt.savefig(fig_roc_path, dpi=300, bbox_inches='tight')
plt.close()
print("[OK] Saved Figure 2.1 to:", fig_roc_path)


[OK] Saved Figure 2.1 to: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn\results\figures\fig_stage2_mlp_roc_curves.png


In [9]:
# Figure: Out-of-Fold Confusion Matrices across all 3 datasets
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for idx, ds_key in enumerate(['breast_cancer', 'heart', 'diabetes']):
    oof_data = oof_predictions_dict[ds_key]
    y_dev = np.array(oof_data["y_true"])
    oof_p = np.array(oof_data["oof_probs"])
    y_pred = (oof_p >= 0.5).astype(int)
    
    cm = confusion_matrix(y_dev, y_pred, labels=[0, 1])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    annot_mat = np.empty_like(cm, dtype=object)
    for i in range(2):
        for j in range(2):
            annot_mat[i, j] = f"{cm[i, j]}\n({cm_norm[i, j]:.1f}%)"
            
    ax = axes[idx]
    sns.heatmap(cm, annot=annot_mat, fmt='', cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 10, 'weight': 'bold'})
    
    if ds_key == 'breast_cancer':
        labels = ['Benign (0)', 'Malignant (1)']
    elif ds_key == 'heart':
        labels = ['No Disease (0)', 'Disease (1)']
    else:
        labels = ['Non-Diab (0)', 'Diabetic (1)']
        
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(f"{ds_key.replace('_', ' ').title()}: OOF CM\n(N={len(y_dev)})", fontweight='bold')

plt.suptitle('Figure 2.2: Out-of-Fold Confusion Matrices for Classical MLP (M1)', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
fig_cm_path = os.path.join(RESULTS_FIGURES_DIR, "fig_stage2_mlp_confusion_matrices.png")
plt.savefig(fig_cm_path, dpi=300, bbox_inches='tight')
plt.close()
print("[OK] Saved Figure 2.2 to:", fig_cm_path)


[OK] Saved Figure 2.2 to: C:\Users\yashw\.gemini\antigravity-ide\scratch\adaptive-hqnn\results\figures\fig_stage2_mlp_confusion_matrices.png


## 10. Stage 2 Acceptance & Validation Summary

We evaluate all programmatic acceptance criteria for Stage 2.


In [10]:
# Stage 2 Programmatic Verification
check_all_datasets_evaluated = set(df_cv_all['dataset'].unique()) == {'breast_cancer', 'heart', 'diabetes'}
check_5_folds_per_dataset = all([len(df_cv_all[(df_cv_all['dataset'] == k) & (df_cv_all['fold'] != 'TEST_ENSEMBLE')]) == 5 for k in ['breast_cancer', 'heart', 'diabetes']])
check_metrics_saved = os.path.exists(metrics_csv_path) and os.path.exists(table_mlp_path)
check_figures_saved = os.path.exists(fig_roc_path) and os.path.exists(fig_cm_path)
check_auc_valid = all([df_cv_all[df_cv_all['dataset'] == k]['roc_auc'].mean() > 0.70 for k in ['breast_cancer', 'heart', 'diabetes']])

all_stage2_passed = all([
    check_all_datasets_evaluated,
    check_5_folds_per_dataset,
    check_metrics_saved,
    check_figures_saved,
    check_auc_valid
])

status_str = "PASS" if all_stage2_passed else "FAIL"

summary_text = f"""
==================================================
STAGE 2 VALIDATION SUMMARY (Classical MLP Baseline)
==================================================

Dataset coverage (3/3): {'PASS' if check_all_datasets_evaluated else 'FAIL'}
5-Fold CV execution (15 folds): {'PASS' if check_5_folds_per_dataset else 'FAIL'}
Performance sanity (AUC > 0.70): {'PASS' if check_auc_valid else 'FAIL'}
Metric logging & Table export: {'PASS' if check_metrics_saved else 'FAIL'}
ROC & CM Figures generated: {'PASS' if check_figures_saved else 'FAIL'}

Overall Stage 2 Status: {status_str}
==================================================
"""

print(summary_text)


STAGE 2 VALIDATION SUMMARY (Classical MLP Baseline)
Dataset coverage (3/3): PASS
5-Fold CV execution (15 folds): PASS
Performance sanity (AUC > 0.70): PASS
Metric logging & Table export: PASS
ROC & CM Figures generated: PASS
Overall Stage 2 Status: PASS
